# ElementalTask-RML
## Notebook 02 — Function-Vector Drift Monitoring

This notebook extends ElementalTask emergence monitoring from task accuracy trajectories to function-vector relationships.

Goal:
- represent tasks as function vectors,
- measure task similarity,
- monitor drift across checkpoints,
- identify whether capability structure stays stable.

**Emergence ≠ magic. Monitor constraints. 📐**

This notebook is designed to run immediately with a lightweight synthetic fallback, while leaving a clearly marked optional cell for real upstream ElementalTask function-vector extraction.

## 0. Setup

The notebook tries to detect whether it is running inside the `ElementalTask-RML` repo. Outputs are saved under:

- `notebooks_rml/figures/`
- `notebooks_rml/results/`
- `notebooks_rml/docs/`

In [ ]:

from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import spearmanr
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False


def find_repo_root(start: Path | None = None) -> Path:
    """Find repo root by walking upward from current directory."""
    start = Path.cwd() if start is None else Path(start)
    markers = ["dataset", "function_vecs", "tasks", "scripts"]
    for p in [start, *start.parents]:
        if any((p / m).exists() for m in markers):
            return p
    return start

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks_rml"
FIG_DIR = NOTEBOOK_DIR / "figures"
RESULTS_DIR = NOTEBOOK_DIR / "results"
DOCS_DIR = NOTEBOOK_DIR / "docs"
for d in [NOTEBOOK_DIR, FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)
print("Figures:", FIG_DIR)
print("Results:", RESULTS_DIR)
print("Docs:", DOCS_DIR)


## 1. Task vocabulary

Notebook 02 begins with a small task vocabulary aligned with Notebook 01:

- atomic/simple tasks,
- math task,
- compositional tasks.

The synthetic fallback creates task vectors that preserve intuitive structure: simple tasks cluster together; compositional tasks bridge their component tasks.

In [ ]:

TASKS = [
    "simple:copying",
    "simple:uppercase",
    "simple:first_letter",
    "math:arithmetic",
    "compositional:copy_then_uppercase",
    "compositional:first_letter_then_uppercase",
]

TASK_KIND = {
    "simple:copying": "atomic",
    "simple:uppercase": "atomic",
    "simple:first_letter": "atomic",
    "math:arithmetic": "math",
    "compositional:copy_then_uppercase": "compositional",
    "compositional:first_letter_then_uppercase": "compositional",
}

CHECKPOINTS = [1000, 5000, 10000, 20000, 50000, 100000]

pd.DataFrame({"task": TASKS, "kind": [TASK_KIND[t] for t in TASKS]})


## 2. Synthetic fallback function-vector generator

This cell lets the notebook run without downloading large models. It creates a controlled stand-in for function vectors.

Interpretation:
- early checkpoints have more noise,
- later checkpoints preserve task geometry more strongly,
- compositional vectors are mixtures of component vectors.

This is not a replacement for upstream function-vector extraction; it is a lightweight monitoring scaffold.

In [ ]:

def l2_normalize(x: np.ndarray, axis: int = -1, eps: float = 1e-12) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)


def make_base_task_vectors(dim: int = 32, seed: int = 42) -> Dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    copy = rng.normal(size=dim)
    uppercase = rng.normal(size=dim)
    first_letter = rng.normal(size=dim)
    arithmetic = rng.normal(size=dim)

    base = {
        "simple:copying": copy,
        "simple:uppercase": uppercase,
        "simple:first_letter": first_letter,
        "math:arithmetic": arithmetic,
        "compositional:copy_then_uppercase": 0.55 * copy + 0.55 * uppercase,
        "compositional:first_letter_then_uppercase": 0.55 * first_letter + 0.55 * uppercase,
    }
    return {k: l2_normalize(v.reshape(1, -1))[0] for k, v in base.items()}


def synthetic_function_vectors(
    tasks: List[str] = TASKS,
    checkpoints: List[int] = CHECKPOINTS,
    dim: int = 32,
    seed: int = 123,
) -> Dict[int, pd.DataFrame]:
    """Return checkpoint -> DataFrame with one function vector per task."""
    rng = np.random.default_rng(seed)
    base = make_base_task_vectors(dim=dim, seed=seed)
    max_ckpt = max(checkpoints)
    out = {}
    for ckpt in checkpoints:
        progress = ckpt / max_ckpt
        noise_scale = 0.65 * (1 - progress) + 0.04
        rows = []
        for task in tasks:
            noisy = base[task] + rng.normal(scale=noise_scale, size=dim)
            vec = l2_normalize(noisy.reshape(1, -1))[0]
            rows.append({"checkpoint": ckpt, "task": task, "kind": TASK_KIND[task], **{f"v{i}": vec[i] for i in range(dim)}})
        out[ckpt] = pd.DataFrame(rows)
    return out

fv_by_checkpoint = synthetic_function_vectors()
print("Synthetic checkpoints:", list(fv_by_checkpoint.keys()))
fv_by_checkpoint[CHECKPOINTS[-1]].head()


## 3. Optional: real upstream ElementalTask FV extraction

The next cell is intentionally commented out. Use it only when your environment has the required model dependencies and you want to extract real vectors from the upstream `function_vecs` API.

A practical workflow is:
1. keep this notebook running with the synthetic fallback,
2. uncomment real extraction once upstream dependencies are installed,
3. save real vectors into `notebooks_rml/results/`,
4. reuse the same similarity and drift cells below.

In [ ]:

# OPTIONAL REAL UPSTREAM FV EXTRACTION — UNCOMMENT ONLY WHEN READY
#
# from function_vecs.extract_function_vecs import extract_function_vector_simple, stack_function_vecs, build_skill_basis
#
# real_tasks = ["simple_icl"]  # Available tasks in upstream README: ["simple_icl", "simple", "textfrct", "math"]
# function_vecs = []
# for task_name in real_tasks:
#     print(f"Extracting function vector for {task_name}...")
#     fv = extract_function_vector_simple(
#         task_name=task_name,
#         model_name="distilgpt2",  # lightweight starter model
#         num_samples=5,
#         device="cpu",
#     )
#     function_vecs.append(fv)
#
# task_matrix = stack_function_vecs(function_vecs)
# skill_basis = build_skill_basis(task_matrix, method="svd", k=min(6, len(function_vecs)))
# print("Task matrix:", task_matrix.V.shape)
# print("Skill basis:", skill_basis.U.shape)


## 4. Pairwise cosine similarity

For each checkpoint, compute cosine similarity between task vectors. Similarity acts as a function-vector geometry monitor.

In [ ]:

def vector_columns(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c.startswith("v")]


def cosine_similarity_matrix(df: pd.DataFrame) -> pd.DataFrame:
    cols = vector_columns(df)
    X = df[cols].to_numpy(dtype=float)
    X = l2_normalize(X, axis=1)
    S = X @ X.T
    return pd.DataFrame(S, index=df["task"], columns=df["task"])

similarities = {ckpt: cosine_similarity_matrix(df) for ckpt, df in fv_by_checkpoint.items()}
final_similarity = similarities[max(similarities)]
final_similarity.round(3)


In [ ]:

# Save final similarity matrix
final_similarity.to_csv(RESULTS_DIR / "02_fv_similarity_final.csv")

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(final_similarity.values, vmin=-1, vmax=1)
ax.set_title("ElementalTask-RML: Function-Vector Similarity (Final Checkpoint)")
ax.set_xticks(range(len(final_similarity.columns)))
ax.set_yticks(range(len(final_similarity.index)))
ax.set_xticklabels(final_similarity.columns, rotation=45, ha="right")
ax.set_yticklabels(final_similarity.index)
fig.colorbar(im, ax=ax, label="cosine similarity")
fig.tight_layout()
fig.savefig(FIG_DIR / "02_function_vector_similarity.png", dpi=200)
plt.show()


## 5. Drift relative to final checkpoint

Measure how much each checkpoint's function-vector geometry differs from the final checkpoint geometry.

A simple drift score is mean absolute difference in pairwise similarity matrices.

In [ ]:

def upper_triangle_values(M: pd.DataFrame) -> np.ndarray:
    arr = M.to_numpy(dtype=float)
    idx = np.triu_indices_from(arr, k=1)
    return arr[idx]

final_vec = upper_triangle_values(similarities[max(similarities)])
rows = []
for ckpt, S in similarities.items():
    cur = upper_triangle_values(S)
    mad = float(np.mean(np.abs(cur - final_vec)))
    if SCIPY_AVAILABLE:
        rho = float(spearmanr(cur, final_vec).correlation)
    else:
        rho = float(pd.Series(cur).corr(pd.Series(final_vec), method="spearman"))
    rows.append({
        "checkpoint": ckpt,
        "fv_similarity_drift_mad": mad,
        "spearman_to_final_geometry": rho,
    })

drift_df = pd.DataFrame(rows).sort_values("checkpoint")
drift_df.to_csv(RESULTS_DIR / "02_fv_drift_scores.csv", index=False)
drift_df


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(drift_df["checkpoint"], drift_df["fv_similarity_drift_mad"], marker="o")
ax.set_title("ElementalTask-RML: Function-Vector Drift")
ax.set_xlabel("Checkpoint order")
ax.set_ylabel("Mean absolute drift from final FV geometry")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_function_vector_drift.png", dpi=200)
plt.show()


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(drift_df["checkpoint"], drift_df["spearman_to_final_geometry"], marker="o")
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title("ElementalTask-RML: FV Geometry Rank Stability")
ax.set_xlabel("Checkpoint order")
ax.set_ylabel("Spearman correlation to final pairwise geometry")
ax.set_ylim(-1.05, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_fv_rank_stability.png", dpi=200)
plt.show()


## 6. Constraint interpretation

Minimal constraint idea:

- compositional task vectors should be closer to their component tasks than to unrelated tasks,
- geometry should become more stable as checkpoints progress.

This section checks two toy constraints for the synthetic fallback:

1. `copy_then_uppercase` should be closer to `copying` and `uppercase` than to `math:arithmetic`.
2. `first_letter_then_uppercase` should be closer to `first_letter` and `uppercase` than to `math:arithmetic`.

In [ ]:

CONSTRAINTS = [
    {
        "composite": "compositional:copy_then_uppercase",
        "components": ["simple:copying", "simple:uppercase"],
        "negative": "math:arithmetic",
    },
    {
        "composite": "compositional:first_letter_then_uppercase",
        "components": ["simple:first_letter", "simple:uppercase"],
        "negative": "math:arithmetic",
    },
]


def constraint_score_for_similarity(S: pd.DataFrame, constraints=CONSTRAINTS) -> Dict[str, float]:
    total = 0
    valid = 0
    detail_rows = []
    for c in constraints:
        comp = c["composite"]
        neg = c["negative"]
        for component in c["components"]:
            total += 1
            component_sim = float(S.loc[comp, component])
            negative_sim = float(S.loc[comp, neg])
            ok = component_sim >= negative_sim
            valid += int(ok)
            detail_rows.append({
                "composite": comp,
                "component": component,
                "negative": neg,
                "component_similarity": component_sim,
                "negative_similarity": negative_sim,
                "valid": ok,
            })
    return {"valid": valid, "total": total, "cgcs_fv": valid / total if total else np.nan, "details": detail_rows}

rows = []
details = []
for ckpt, S in similarities.items():
    score = constraint_score_for_similarity(S)
    rows.append({"checkpoint": ckpt, "valid": score["valid"], "total": score["total"], "cgcs_fv": score["cgcs_fv"]})
    for row in score["details"]:
        details.append({"checkpoint": ckpt, **row})

fv_cgcs_df = pd.DataFrame(rows).sort_values("checkpoint")
fv_constraint_details = pd.DataFrame(details).sort_values(["checkpoint", "composite", "component"])
fv_cgcs_df.to_csv(RESULTS_DIR / "02_fv_constraint_scores.csv", index=False)
fv_constraint_details.to_csv(RESULTS_DIR / "02_fv_constraint_details.csv", index=False)
fv_cgcs_df


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(fv_cgcs_df["checkpoint"], fv_cgcs_df["cgcs_fv"], marker="o")
ax.set_title("ElementalTask-RML: FV Constraint Score")
ax.set_xlabel("Checkpoint order")
ax.set_ylabel("CGCS-FV")
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_fv_constraint_score.png", dpi=200)
plt.show()


## 7. Compact monitoring summary

This table joins drift and constraint scores into one checkpoint monitor.

In [ ]:

summary = drift_df.merge(fv_cgcs_df[["checkpoint", "cgcs_fv"]], on="checkpoint", how="left")
summary["monitor_flag"] = np.select(
    [
        summary["cgcs_fv"] < 0.75,
        summary["fv_similarity_drift_mad"] > summary["fv_similarity_drift_mad"].median(),
    ],
    ["constraint-drift", "geometry-drift"],
    default="stable-or-improving",
)
summary.to_csv(RESULTS_DIR / "02_fv_monitor_summary.csv", index=False)
summary


## 8. Export mini report

Write a short markdown summary for the repo README or notebook index.

In [ ]:

report = """# Notebook 02 — Function-Vector Drift Monitoring

This notebook adds a lightweight function-vector geometry monitor for ElementalTask-RML.

## Outputs

- `figures/02_function_vector_similarity.png`
- `figures/02_function_vector_drift.png`
- `figures/02_fv_rank_stability.png`
- `figures/02_fv_constraint_score.png`
- `results/02_fv_similarity_final.csv`
- `results/02_fv_drift_scores.csv`
- `results/02_fv_constraint_scores.csv`
- `results/02_fv_constraint_details.csv`
- `results/02_fv_monitor_summary.csv`

## Initial interpretation

Function-vector similarity gives a second monitoring layer beyond accuracy curves.

A stable training trajectory should preserve task geometry: similar atomic tasks cluster, compositional tasks remain close to their component tasks, and pairwise geometry drifts less as checkpoint order increases.

Pipeline:

`checkpoint → function-vector geometry → similarity drift → constraint score`

Emergence ≠ magic. Monitor constraints. 📐
"""

out = DOCS_DIR / "02_function_vector_drift.md"
out.write_text(report)
print(out)
print(report)


## 9. Optional Colab zip download

Uncomment the next cell in Colab to zip Notebook 02 figures, results, and docs for download.

In [ ]:

# OPTIONAL COLAB DOWNLOAD — UNCOMMENT IN COLAB
#
# import zipfile
# from google.colab import files
#
# EXPORT_NAME = "elementaltask_rml_notebook02_outputs.zip"
# export_paths = []
# for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
#     export_paths.extend(folder.glob("02_*"))
#
# with zipfile.ZipFile(EXPORT_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
#     for path in export_paths:
#         zf.write(path, arcname=str(path.relative_to(NOTEBOOK_DIR)))
#
# files.download(EXPORT_NAME)


## 10. Conclusion

Notebook 01 monitored emergence-order stability from accuracy trajectories.

Notebook 02 adds function-vector geometry:

- pairwise task similarity,
- drift relative to final geometry,
- constraint score for component/composite relationships,
- compact checkpoint monitor flags.

Future work:

- replace synthetic fallback with real upstream FV extraction,
- compare multiple models,
- align FV geometry with emergence-rank stability,
- add RML lane visualizations for task trajectories.